# 01 - Data quality and cleaning

Vietnamese property listings from Hugging Face: schema checks, missing values, duplicates and the creation of the analytical datasets.

* Data source: https://huggingface.co/datasets/tinixai/vietnam-real-estates
* Team:
* Team members:

**Inputs:** `dataset/vietnam_real_estates.parquet` (downloaded on first run)  
**Outputs:** `data/processed/clean_listings.parquet`, `data/processed/priced_listings.parquet`  
**Next:** `02_eda_price-analysis.ipynb`


**Dataset information**

| Column | Meaning |
|---|---|
| `name` | Listing title |
| `description` | Listing description text |
| `property_type_name` | Property category (apartment, house, land, ...) |
| `province_name` | Province / city |
| `district_name` | District |
| `ward_name` | Ward |
| `street_name` | Street |
| `project_name` | Development project name (if any) |
| `price` | Asking price in VND (`null` if not disclosed) |
| `area` | Floor / land area in m² |
| `floor_count` | Number of floors |
| `frontage_width` | Frontage width (m) |
| `house_depth` | Depth (m) |
| `road_width` | Front road width (m) |
| `bedroom_count` | Number of bedrooms |
| `bathroom_count` | Number of bathrooms |
| `house_direction` | Facing direction |
| `balcony_direction` | Balcony facing direction |
| `published_at` | Listing publish timestamp |


**Research objectives**

Purpose: understand how asking prices of Vietnamese property listings vary by location and property type, and how reliable the data is for those comparisons.

1. **Data quality:** How complete, duplicated and valid are the listings, and which rows are kept for price analysis?
2. **Price patterns:** How do asking price and price per m² differ by province, district and property type among listings with disclosed price and area? Do the conclusions hold after removing duplicates?
3. **Associated factors:** Which attributes (area, frontage, floors, bedrooms, bathrooms) are most associated with asking price and price per m²? (Spearman on raw values, Pearson on log10.)

Assumption: every listing is treated as a property for sale, and `price` is the price the seller asks for. Results only cover listings that show a price.


## 0. SET UP


In [ ]:
# uncmt if using colab
# import sys
# if "google.colab" in sys.modules:
#     !git clone <url-repo-github> /content/repo
#     %cd /content/repo/notebooks
#     !pip install -q -r ../requirements.txt

In [ ]:
import sys

# Notebooks live in notebooks/, so the repo root is one level up.
sys.path.insert(0, "..")

import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

from eda_utils.config import (
    CATEGORY_COLUMNS,
    CLEAN_PARQUET,
    EXPECTED_COLUMNS,
    PRICED_PARQUET,
    RANDOM_STATE,
    RAW_DIR,
    RAW_PARQUET,
)
from eda_utils.helpers import (
    clean_data,
    format_price_table,
    pairwise_corr,
    plot_box,
    prepare_priced_data,
    price_summary,
    title_with_n,
)

np.random.seed(RANDOM_STATE)

pd.options.mode.copy_on_write = True
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 12

In [ ]:
if RAW_PARQUET.exists():
    print("Reading dataset...")
    df = pd.read_parquet(RAW_PARQUET)
else:
    print("Downloading dataset...")
    ds = load_dataset("tinixai/vietnam-real-estates", cache_dir=str(RAW_DIR))
    df = ds["train"].to_pandas()

    RAW_DIR.mkdir(parents=True, exist_ok=True)
    df.to_parquet(RAW_PARQUET, index=False)
print("Rows, columns:", df.shape)

In [ ]:
missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
extra = [c for c in df.columns if c not in EXPECTED_COLUMNS]
assert not missing, f"Missing columns: {missing}"
assert not extra, f"Unexpected columns: {extra}"
print(f"Schema OK: {len(EXPECTED_COLUMNS)} expected columns present.")

## 1. DATA CLEANING

Conservative cleaning: flag issues, keep all rows in `df_clean`, and build `df_priced` as a separate analytical subset.


### 1.1. Initial data inspection


In [ ]:
print("Shape:", df.shape)
print(df.dtypes)
display(df.head())

In [ ]:
def memory_gb(frame: pd.DataFrame) -> float:
    return frame.memory_usage(deep=True).sum() / (1024 ** 3)

print(f"Memory before category conversion: {memory_gb(df):.3f} GB")

for col in CATEGORY_COLUMNS:
    df[col] = df[col].astype("category")

print(f"Memory after category conversion: {memory_gb(df):.3f} GB")

### 1.2. Missing value analysis

Missing `price` means the asking price was not disclosed. Missing `project_name` often means the listing is not in a named project. Missing directions and dimensions mean the listing did not provide that detail. Rows with missing values are retained.


In [ ]:
miss = df.isna().sum().rename("n_missing").to_frame()
miss["pct"] = 100 * miss["n_missing"] / len(df)
miss = miss.sort_values("pct", ascending=False)
display(miss)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(miss.index.astype(str), miss["pct"], color="steelblue")
ax.set_xlabel("% missing")
ax.set_title(title_with_n("Missing percentage by column", "df", len(df)))
plt.tight_layout()
plt.show()

In [ ]:
type_order = df["property_type_name"].value_counts().index
miss_by_type = (
    df.drop(columns="property_type_name").isna()
    .groupby(df["property_type_name"], observed=True).mean()
    .reindex(type_order)
)
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(miss_by_type, annot=True, fmt=".0%", cmap="Reds", ax=ax)
ax.set_title(title_with_n("Null rate by property type", "df", len(df)))
plt.tight_layout()
plt.show()

Listings without a valid price (price missing or ≤ 0, or area missing or ≤ 0) are excluded from `df_priced`. The tables below show whether these listings are spread evenly across property types, provinces and publication months.

In [ ]:
excluded = ~(df["price"].gt(0) & df["area"].gt(0))
print(f"Share of listings excluded from df_priced: {excluded.mean():.1%}")

month = pd.to_datetime(df["published_at"], errors="coerce").dt.to_period("M")


def excluded_rate(by):
    t = excluded.groupby(by, observed=True).agg(n="size", excluded_rate="mean")
    return t[t["n"] >= 100]


fmt = {"excluded_rate": "{:.1%}"}
print(title_with_n("Excluded share by property type", "df", len(df)))
display(excluded_rate(df["property_type_name"]).sort_values("n", ascending=False).style.format(fmt))
print(title_with_n("Excluded share by province (top 15 by n)", "df", len(df)))
display(excluded_rate(df["province_name"]).sort_values("n", ascending=False).head(15).style.format(fmt))
print(title_with_n("Excluded share by publication month", "df", len(df)))
display(excluded_rate(month).style.format(fmt))

### 1.3. Data type validation and geography / direction checks


In [ ]:
numeric_cols = [
    "price", "area", "floor_count", "frontage_width", "house_depth",
    "road_width", "bedroom_count", "bathroom_count",
]
pub = pd.to_datetime(df["published_at"], errors="coerce")
print("published_at parse nulls:", int(pub.isna().sum()))
print("published_at min/max:", pub.min(), pub.max())
now = pd.Timestamp.now("UTC").tz_localize(None)
pub_naive = pub.dt.tz_localize(None) if pub.dt.tz is not None else pub
print("published_at later than notebook run timestamp:", int((pub_naive > now).sum()))


In [ ]:
print("province nunique:", df["province_name"].nunique(dropna=False))
print("district nunique:", df["district_name"].nunique(dropna=False))
print("Top 20 provinces:\n", df["province_name"].value_counts().head(20))
dist_prov = (
    df.dropna(subset=["district_name"])
    .groupby("district_name", observed=True)["province_name"]
    .nunique()
)
print("district_name values spanning >1 province:", int((dist_prov > 1).sum()))


In [ ]:
print(title_with_n("house_direction value counts", "df", len(df)))
display(df["house_direction"].value_counts(dropna=False))
print(title_with_n("balcony_direction value counts", "df", len(df)))
display(df["balcony_direction"].value_counts(dropna=False))


### 1.4. Cleaning and analytical dataset creation

Derived columns added to `df_clean` (all 19 originals retained):

| Column | Definition |
|---|---|
| `is_exact_dup` | Exact duplicate over the 19 original columns |
| `is_soft_dup` | Soft-key duplicate |
| `soft_key` | Hash used to identify soft duplicates |
| `flag_nonpositive_price` | `price <= 0` (null → False) |
| `flag_nonpositive_area` | `area <= 0` (null → False) |
| `flag_negative_floor_count` | `floor_count < 0` |
| `flag_negative_bedroom_count` | `bedroom_count < 0` |
| `flag_negative_bathroom_count` | `bathroom_count < 0` |
| `flag_extreme_area` | `area` above 99.9th pct of positive areas |
| `flag_extreme_floor_count` | above 99.9th pct of non-negative values |
| `flag_extreme_bedroom_count` | above 99.9th pct of non-negative values |
| `flag_extreme_bathroom_count` | above 99.9th pct of non-negative values |

`df_priced` keeps listings with `price > 0` and `area > 0` (sale-market assumption), and adds `price_per_m2` and `flag_extreme_price_per_m2` only.


In [ ]:
df_clean = clean_data(df)
df_priced = prepare_priced_data(df_clean)

assert len(df_clean) == len(df) and df_clean.index.equals(df.index)
assert (df_priced["price"] > 0).all() and (df_priced["area"] > 0).all()
assert "price_per_m2" not in df_clean.columns

n0 = len(df)
n_price_ok = int(df_clean["price"].gt(0).sum())
n_priced = len(df_priced)
print(f"df / df_clean: {n0:,} rows")
print(f"after excluding missing/nonpositive price: {n_price_ok:,} "
      f"(excluded {n0 - n_price_ok:,}, {(n0 - n_price_ok) / n0:.2%})")
print(f"after excluding missing/nonpositive area (= df_priced): {n_priced:,} "
      f"(excluded {n_price_ok - n_priced:,}, {(n_price_ok - n_priced) / n0:.2%})")

In [ ]:
print(f"Share of df_priced rows with price < 10,000,000 VND: {(df_priced['price'] < 10_000_000).mean():.2%}")
print("df_priced price percentiles (1, 50, 99):")
print(df_priced["price"].quantile([0.01, 0.50, 0.99]))

#### Duplicate report and extreme tails

Exact and soft duplicates are reported only. Nothing is deleted.

In [ ]:
for name, col in [("Exact", "is_exact_dup"), ("Soft", "is_soft_dup")]:
    print(f"{name} duplicates (keep=first): {df_clean[col].sum():,} ({df_clean[col].mean():.2%})")

cluster_sizes = df_clean["soft_key"].value_counts()
top_clusters = cluster_sizes[cluster_sizes > 1].head(10)
examples = (
    df_clean[df_clean["soft_key"].isin(top_clusters.index)]
    .drop_duplicates("soft_key")
    .assign(cluster_size=lambda d: d["soft_key"].map(top_clusters))
    .sort_values("cluster_size", ascending=False)
)
print(title_with_n("10 largest soft-duplicate clusters (one example row each)", "df_clean", len(df_clean)))
display(examples[["cluster_size", "name", "province_name", "district_name", "price", "area"]])

In [ ]:
cols_show = ["name", "province_name", "price", "area", "price_per_m2"]
for col in ["area", "price_per_m2"]:
    q = df_priced[col].quantile([0.01, 0.99, 0.999])
    print(f"{col} percentiles 1/99/99.9:\n", q)
    lo, hi = q.loc[0.01], q.loc[0.999]
    below = df_priced[df_priced[col] < lo]
    above = df_priced[df_priced[col] > hi]
    print(f"5 random rows below 1st pct ({col}):")
    display(below.sample(n=min(5, len(below)), random_state=RANDOM_STATE)[cols_show])
    print(f"5 random rows above 99.9th pct ({col}):")
    display(above.sample(n=min(5, len(above)), random_state=RANDOM_STATE)[cols_show])

### 1.5. Save analytical datasets

Persist `df_clean` (all rows, flags kept) and `df_priced` (valid price and area) so that notebook 02 and every modelling notebook start from the same data.

In [ ]:
CLEAN_PARQUET.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(CLEAN_PARQUET, index=False)
df_priced.to_parquet(PRICED_PARQUET, index=False)
print(f"Saved {len(df_clean):,} rows -> {CLEAN_PARQUET.name}")
print(f"Saved {len(df_priced):,} rows -> {PRICED_PARQUET.name}")